<h1>Heat of Formation Calculator Using Benson Group Increments</h1>
<h4>Designed for Chemistry C450/C540 at Indiana University Bloomington by Prashant Kumar and Dr. Nicola L. B. Pohl</h4><br>
September 12, 2024 Version<br>
<ul>
<li>This notebook is designed to calculate approximate heats of formation of organic molecules based on the idea of Benson Group Increments: Cohen & Benson, <em>Chem. Rev.</em> <b>1993</b>, <em>93</em>, 2419.</li>
<li>The notebook takes values from a user's file titled "increment_correction_table.csv" that is in the same folder as this notebook file. This csv file contains one column titled "Benson Group Increment" populated by Benson Group increment types and a second column titled "Delta_Hf kJ/mol" populated by numerical values in kJ/mol. Research is ongoing to update values and add increments to better describe a range of molecules; the user can decide which increments to use in the calculator.</li>
<li>The Benson Group Increments are rendered into compact buttons that the user can click on to select. The value associated with that button is automatically added to the total displayed below the buttons. The user needs to decide which increments and corrections are needed based on the molecule of interest to make the entire process of estimation transparent.</li> 
</ul>

## Archived — generation 2, frozen 2026-09-15 (WP7)

This is the **generation-2** notebook (modular per-category CSVs, its own
`parse_value`/`load_increment_data`/`get_value_dicts`), archived when
generation 4 (the live `Benson Increments Calculator.ipynb` at the repository
root) replaced it. It is a historical record, not a maintained product — see
`Plan/00-context.md`'s history table if you have the planning set.

It reads a **frozen copy** of the data, not the live `CSV_data_files/`:
[`Archives/CSV_data_files_gen2/`](CSV_data_files_gen2/), snapshotted from
`CSV_data_files/` as it stood on 2026-09-15, the same way
[`Archives/increment_correction_table.csv`](../increment_correction_table.csv)
freezes generation 1's data. Run it the way the live notebook has always been
run — started from the **repository root**, not from inside `Archives/` —
because the cell below names the frozen folder by its path from there.

Needs `pandas` and `ipywidgets` (`pip install -r requirements.txt` at an
older commit, or `pip install pandas ipywidgets` directly — this generation's
own `requirements.txt` entry retired with it).


In [ ]:
# --- Constants ---
# Conversion factor from kilojoules to kilocalories (source: NIST)
KJ_TO_KCAL = 0.239006

# --- Imports ---
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import functools
import os
import glob

# --- Helper Functions ---
def parse_value(v):
    # A published range is written '1.05 to 1.76' and is averaged. The
    # separator is a word because a hyphen also starts a negative number,
    # and telling those apart is the one thing this had to be careful about.
    if isinstance(v, str):
        v = v.strip()
        if ' to ' in v:
            low, high = v.split(' to ')
            return (float(low) + float(high)) / 2
        return float(v)
    return float(v)

# --- Data Loading ---
def load_increment_data(folder_path):
    """Load all CSV files in the specified folder and return a dict of {filename: df}.
    
    This function implements the modular CSV system where each Benson group category
    is stored in a separate CSV file. The system automatically discovers and loads
    all valid CSV files in the specified folder.
    
    Args:
        folder_path (str): Path to the folder containing CSV files
        
    Returns:
        dict: Dictionary mapping filename (without .csv extension) to pandas DataFrame
        
    Raises:
        ValueError: If no valid CSV files are found in the folder
    """
    data_dict = {}
    csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            df.columns = df.columns.str.strip()
            df.fillna(0, inplace=True)
            filename = os.path.basename(csv_file).replace('.csv', '')
            data_dict[filename] = df
        except Exception as e:
            print(f"Warning: Could not load {csv_file}: {e}")
    if not data_dict:
        raise ValueError("No valid CSV files found in the folder.")
    return data_dict

def get_value_dicts(data_dict):
    """Extract value dictionaries from each DataFrame, assuming 2 columns: group and value.
    
    This function processes the loaded CSV data and creates category dictionaries.
    Each CSV file becomes a category with buttons for each valid group-value pair.
    The system expects exactly 2 columns per CSV file.
    
    Args:
        data_dict (dict): Dictionary of {filename: DataFrame} from load_increment_data()
        
    Returns:
        dict: Dictionary mapping category names to {group_name: value} dictionaries
    """
    categories = {}
    for category_name, df in data_dict.items():
        if len(df.columns) != 2:
            print(f"Warning: {category_name}.csv does not have exactly 2 columns. Skipping.")
            continue
        group_col, value_col = df.columns
        cat_dict = {}
        for _, row in df.iterrows():
            group = row[group_col]
            value = row[value_col]
            if pd.notna(group) and pd.notna(value) and group != '' and value != '':
                cat_dict[str(group)] = value
        categories[category_name] = cat_dict
    return categories

# --- Widget Creation ---
def create_increment_button(label, value, color, on_click_fn):
    """Create a compact button for a group increment or correction."""
    if label == '' or value == '':
        return None
    button = widgets.Button(description=f"{label}", layout=widgets.Layout(width="125px", height="25px", padding="0px 0px 0px 0px"))
    button.style.button_color = color
    button.on_click(lambda b: on_click_fn(value, label))
    return button

def create_button_grid(buttons, columns=6):
    """Create a grid layout for a list of buttons."""
    return widgets.GridBox([btn for btn in buttons if btn], layout=widgets.Layout(grid_template_columns=f"repeat({columns}, 150px)", grid_gap="2px"))

# Define a list of colors for dynamic assignment
COLORS = ['lightblue', 'lightpink', 'lightgreen', 'lightcoral', 'lightyellow', 'lightcyan', 'lightgray']

def format_title(category_name):
    """Format category name for display, properly handling chemical acronyms.
    
    This function converts CSV filenames to user-friendly tab titles by:
    1. Removing numbered prefixes (e.g., "01_" from "01_CH_Groups")
    2. Converting underscores to spaces
    3. Keeping chemical acronyms uppercase (CH, CHO, CHNO, etc.)
    4. Properly capitalizing other words
    
    Args:
        category_name (str): CSV filename without .csv extension (e.g., "01_CH_Groups")
        
    Returns:
        str: Formatted title for display (e.g., "CH Groups")
    """
    # Remove number prefix if present (e.g., "01_CH_Groups" -> "CH_Groups")
    if category_name[0].isdigit() and '_' in category_name:
        # Find the first underscore after the number
        parts = category_name.split('_', 1)
        if len(parts) > 1 and parts[0].isdigit():
            category_name = parts[1]
    
    words = category_name.replace('_', ' ').split()
    formatted_words = []
    
    for word in words:
        # Keep common chemical acronyms and elements uppercase
        chemical_acronyms = ['CH', 'CHO', 'CHNO', 'CHON', 'CO', 'OH', 'NH', 'SH', 'NO', 'SO', 'SI', 'CL', 'BR', 'I', 'F', 'A']
        if word.upper() in chemical_acronyms or (len(word) <= 3 and word.isalpha() and word.upper() == word):
            formatted_words.append(word.upper())
        else:
            formatted_words.append(word.capitalize())
    
    return ' '.join(formatted_words)

# --- State & UI ---
total_heat_kj = 0.0
increment_history = []
selected_button_labels = []

# Add a history panel output widget
history_panel = widgets.VBox()

total_label_kj = widgets.Label()
total_label_kcal = widgets.Label()
selected_buttons_label = widgets.Label()

def update_labels():
    total_label_kj.value = f"Standard heat of formation: {total_heat_kj:.2f} kJ/mol"
    total_label_kcal.value = f"Standard heat of formation: {total_heat_kj * KJ_TO_KCAL:.2f} kcal/mol"
    selected_buttons_label.value = f"Pressed buttons: {', '.join(selected_button_labels) if selected_button_labels else 'None'}"

def update_history_panel():
    """Redraw the history panel with current increment history and remove buttons."""
    if not increment_history:
        history_panel.children = [widgets.HTML('<b>No increments added yet.</b>')]
    else:
        items = []
        for idx, (label, value) in enumerate(zip(selected_button_labels, increment_history)):
            remove_btn = widgets.Button(description='Remove', layout=widgets.Layout(width='70px', height='22px'))
            remove_btn.on_click(functools.partial(remove_history_item, idx))
            item_box = widgets.HBox([widgets.Label(f'{label}: {value:+.2f} kJ/mol'), remove_btn])
            items.append(item_box)
        history_panel.children = items

def remove_history_item(index, button=None):
    """Remove an item from history by index and update state/UI."""
    global total_heat_kj
    if 0 <= index < len(increment_history):
        total_heat_kj -= increment_history[index]
        del increment_history[index]
        del selected_button_labels[index]
        update_labels()
        update_history_panel()

# --- Event Handlers ---
def add_to_total(value, label):
    global total_heat_kj
    parsed = parse_value(value)
    total_heat_kj += parsed
    increment_history.append(parsed)
    selected_button_labels.append(label)
    update_labels()
    update_history_panel()

def undo_last_action(button):
    global total_heat_kj
    if increment_history and selected_button_labels:
        last_value = increment_history.pop()
        total_heat_kj -= last_value
        selected_button_labels.pop()
        update_labels()
        update_history_panel()

def reset_all(button):
    global total_heat_kj, increment_history, selected_button_labels
    total_heat_kj = 0.0
    increment_history = []
    selected_button_labels = []
    update_labels()
    update_history_panel()

# --- Main UI Assembly ---
def main():
    """Main function that assembles the complete Benson Increments Calculator UI.
    
    This function implements the dynamic modular system:
    1. Loads all CSV files from Archives/CSV_data_files_gen2/ - a frozen snapshot,
       archived alongside this notebook at WP7 (2026-09-15), not the live
       CSV_data_files/ the current generation reads
    2. Creates categories and buttons dynamically from CSV data
    3. Generates tab titles with proper chemical acronym formatting
    4. Assembles the complete interactive interface

    Run this notebook from the repository root - the path below is relative
    to there, the same way the live notebook has always documented running it.
    """
    data_dict = load_increment_data('Archives/CSV_data_files_gen2/')
    categories = get_value_dicts(data_dict)
    
    tab_children = []
    tab_titles = []
    color_idx = 0
    
    for category_name, cat_dict in categories.items():
        color = COLORS[color_idx % len(COLORS)]
        color_idx += 1
        buttons = [create_increment_button(label, value, color, add_to_total) for label, value in cat_dict.items()]
        layout = create_button_grid(buttons)
        tab_children.append(layout)
        # Format the title properly handling chemical acronyms
        title = format_title(category_name)
        tab_titles.append(title)
    
    undo_button = widgets.Button(description="Undo the last addition", layout=widgets.Layout(width="440px", height="25px"))
    undo_button.on_click(undo_last_action)
    reset_button = widgets.Button(description="Reset all selections", layout=widgets.Layout(width="440px", height="25px"))
    reset_button.on_click(reset_all)
    button_container = widgets.HBox([undo_button, reset_button])
    
    tab = widgets.Tab()
    tab.children = tab_children
    for i, title in enumerate(tab_titles):
        tab.set_title(i, title)
    
    update_labels()
    update_history_panel()
    display(tab)
    display(button_container)
    display(total_label_kj)
    display(total_label_kcal)
    display(selected_buttons_label)
    display(history_panel)

main()